In [ ]:
pip install transformers==4.43.4 trl==0.8.6 peft==0.11.1 accelerate bitsandbytes

In [ ]:
#import kagglehub

# Download latest version
#path = kagglehub.dataset_download("orvile/english-to-turkish-sentence-pairs")

#print("Path to dataset files:", path)

In [ ]:
import pandas as pd
import os

#print(os.listdir(path))  # see available files


#file_path = path + "/Sentence pairs in English-Turkish - 2025-05-06.tsv"

#df = pd.read_csv(file_path, sep="\t", engine="python", on_bad_lines="skip")

#df.head()

In [ ]:
df = pd.read_csv("/kaggle/input/datasets/buketsak/data-opus/data.csv")
df.head()

In [ ]:
len(df)

In [ ]:
df.columns

In [ ]:
from sklearn.model_selection import train_test_split

# First split: Train (80%) and Temp (20%)
train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

# Second split: Validation (10%) and Test (10%)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    shuffle=True
)

print("Train size:", len(train_df))
print("Validation size:", len(val_df))
print("Test size:", len(test_df))

In [ ]:
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

In [ ]:
print(len(train_df), len(val_df), len(test_df))

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
token = user_secrets.get_secret("token")

In [ ]:
from huggingface_hub import login
login(token) #token needs to be provided

In [ ]:
from datasets import Dataset

def format_example(row):
    return {
        "messages": [
            {"role": "system", "content": "You are a professional English to Turkish translator."},
            {"role": "user", "content": f"Translate this sentence from English to Turkish:\n\nEnglish: {row["Let's try something."]}"},
            {"role": "assistant", "content": row["Bir şeyler deneyelim!"]}
        ]
    }

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

train_dataset = train_dataset.map(format_example)
val_dataset = val_dataset.map(format_example)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

In [ ]:
import transformers
print(transformers.__version__)

In [ ]:
def apply_chat_template(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

train_dataset = train_dataset.map(apply_chat_template)
val_dataset = val_dataset.map(apply_chat_template)

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], #maybe remove mlp part:  "gate_proj", "up_proj", "down_proj"
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir="/kaggle/working/llama-mt-lora",
    per_device_train_batch_size=1,
    #per_device_eval_batch_size=4,
    gradient_accumulation_steps=4, #  optim="paged_adamw_32bit",
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=50,
    evaluation_strategy="no",
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    fp16=True,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    args=training_args,
    packing=False,
    dataset_text_field="text",
    max_seq_length = 256
)

In [ ]:
#trainer.train()

In [ ]:
trainer.model.save_pretrained("/kaggle/working/llama-mt-lora")
tokenizer.save_pretrained("/kaggle/working/llama-mt-lora")